In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [2]:
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
    .getOrCreate()


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a9b4db5d-ccc7-4546-8b7e-d961646e24b6;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 954ms :: artifacts dl 11ms
	:: modules in us

In [3]:
users_df = spark.read.csv("data/user_table.csv", header=True, inferSchema=True)

In [4]:
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True),
])

In [5]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud_detection") \
    .load()

In [6]:
parsed_stream = kafka_stream.select(from_json(col('value').cast('string'), tx_schema).alias("tx")).select("tx.*")
fraud_stream = parsed_stream.filter(col('amount') > 10000.0)
normal_stream = parsed_stream.filter((col('amount') > 5000.0) & (col('amount') < 10000.0))

In [7]:
enriched_data1 = fraud_stream.join(users_df, "userId")
enriched_data2 = normal_stream.join(users_df, "userId")

In [8]:
output_stream2 = enriched_data2 \
    .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

output_stream1 = enriched_data1 \
    .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

26/06/19 18:27:31 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [9]:
query1 = output_stream1.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud_notification") \
 .option("checkpointLocation", "/workspace/checkpoints12") \
 .start()

query2 = output_stream2.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "normal_notification") \
 .option("checkpointLocation", "/workspace/checkpoints23") \
 .start()

query1.awaitTermination()
query2.awaitTermination()

26/06/19 18:27:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 18:27:54 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 18:27:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 18:27:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "

KeyboardInterrupt: 

In [10]:
for q in spark.streams.active:
    q.stop()

In [11]:
import shutil

shutil.rmtree("/workspace/checkpoints12", ignore_errors=True)
shutil.rmtree("/workspace/checkpoints23", ignore_errors=True)